## LOGISTIC REGRESSION

```
1. Data Exploration:
a. Load the dataset and perform exploratory data analysis (EDA).
b. Examine the features, their types, and summary statistics.
c. Create visualizations such as histograms, box plots, or pair plots to visualize the distributions and relationships between features.
Analyze any patterns or correlations observed in the data.

2. Data Preprocessing:
a. Handle missing values (e.g., imputation).
b. Encode categorical variables.

3. Model Building:
a. Build a logistic regression model using appropriate libraries (e.g., scikit-learn).
b. Train the model using the training data.

4. Model Evaluation:
a. Evaluate the performance of the model on the testing data using accuracy, precision, recall, F1-score, and ROC-AUC score.
Visualize the ROC curve.

5. Interpretation:
a. Interpret the coefficients of the logistic regression model.
b. Discuss the significance of features in predicting the target variable (survival probability in this case).

6. Deployment with Streamlit:
In this task, you will deploy your logistic regression model using Streamlit. The deployment can be done locally or online via Streamlit Share. Your task includes creating a Streamlit app in Python that involves loading your trained model and setting up user inputs for predictions. 

(optional)For online deployment, use Streamlit Community Cloud, which supports deployment from GitHub repositories. 

```

## Answer
```
1. Data Exploration:
a. Load the dataset and perform exploratory data analysis (EDA).
b. Examine the features, their types, and summary statistics.
c. Create visualizations such as histograms, box plots, or pair plots to visualize the distributions and relationships between features.
Analyze any patterns or correlations observed in the data.
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

#Loading the data to the enviroment
df = pd.read_csv("diabetes.csv")

# making another dataset copy
dataset = df.copy()

#Getting information about the data
print("\n<-----INFO------>\n")
print(dataset.info())

print("\n<-----DESCRIBE NUMERICAL VARIABLES------>\n")
print(dataset.describe())

print("\n<-----DESCRIBE BOTH NUMERICAL AND CATEGORICAL------>\n")
print(dataset.describe(include='all'))

print("\n<-----MISSING VALUES------>\n")
print(dataset.isnull().sum())



In [ ]:
#univariate Analysis

numerical_cols = dataset.select_dtypes(include=["number"]).columns.tolist()
numerical = [col for col in  numerical_cols if not col == 'Outcome']
categorical_col = 'Outcome' # it's a categorical value eg,0, 1
print(numerical)

sns.set(style="whitegrid")
for col in numerical:
    plt.figure(figsize=(18,8))

    plt.subplot(1,2,1)
    plt.title(f"Histogram for {col}")
    sns.histplot(dataset[col], kde=True, bins=20)

    plt.subplot(1,2,2)
    plt.title(f"Box plot for {col}")
    sns.boxplot(dataset[col])

    plt.show()
    

    

In [ ]:
# scatterplot between all possible numerical cols
for i, x_col in enumerate(numerical):
    for y_col in numerical[i+1:]:
          if x_col < y_col:
            plt.figure(figsize=(20,8))
            plt.title(f"The Scatterplot between {x_col} and {y_col} by NSP")
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.scatter(x=x_col, y=y_col, data=dataset)
            plt.show()

```
2. Data Preprocessing:
a. Handle missing values (e.g., imputation).
b. Encode categorical variables.
```

In [ ]:
# Some columns in the table have unrealistic values like 0, in the Bloodpressure, skinthickness, insulin and BMI
cols_with_zero_invalid = ['BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
dataset[cols_with_zero_invalid] = dataset[cols_with_zero_invalid].replace(0,np.nan)

# Filling the missing values
for col in cols_with_zero_invalid:
    dataset.fillna(dataset[col].median(),inplace=True)

print(dataset)
    

In [ ]:
# Boxplot and histogram for the modified columns 
for col in cols_with_zero_invalid:
    plt.figure(figsize=(18,8))

    plt.subplot(1,2,1)
    sns.histplot(dataset[col],kde=True)
    plt.title(f"Histogram for modified {col}")

    plt.subplot(1,2,2)
    plt.title(f"new Boxplot for modified {col}")
    sns.boxplot(dataset[col])

    plt.show()

In [ ]:

numeric_cols = [col for col in dataset.columns.tolist() if not col == 'Outcome']

for col in numeric_cols:
    Q1 = dataset[col].quantile(0.25)
    Q3 = dataset[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Apply capping directly on dataset
    dataset[col] = dataset[col].clip(lower=lower_bound, upper=upper_bound)


In [ ]:
# Again checking whether outliers are there or not by using box plot
print("\n<-----Boxplot after removing all the outliers------>\n")
for col in numerical_cols:
    plt.figure(figsize=(18,8))
    sns.boxplot(dataset[col])
    

    plt.show()

```
3. Model Building:
a. Build a logistic regression model using appropriate libraries (e.g., scikit-learn).
b. Train the model using the training data.
```

In [ ]:
# Dividing Data into features and target
X = dataset[[col for col in dataset.columns.tolist() if not col == 'Outcome']]
y = dataset['Outcome']

In [ ]:
#Train and test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42,stratify=y)


In [ ]:
# Feature Scaling(standardization)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Using GridSearchCV with Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

log_reg = LogisticRegression()

parameters = {
    'penalty':['l1', 'l2', 'elasticnet'],
    'C':[0.01, 0.1, 1, 2,3,4,5,10,100],
    'max_iter':[100,200,300,400,500],
    'solver':['liblinear','saga']
}

# Using GridSearchCV
logistic_regressor = GridSearchCV(log_reg, parameters, scoring="accuracy", cv=5 )
logistic_regressor.fit(X_train,y_train)


In [ ]:
print("best parameters", logistic_regressor.best_params_)
print("Best CV Accuracy:", logistic_regressor.best_score_)

```
4. Model Evaluation:
a. Evaluate the performance of the model on the testing data using accuracy, precision, recall, F1-score, and ROC-AUC score.
Visualize the ROC curve.
```

In [ ]:
# Evaluating model
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

best_model = logistic_regressor.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Accuracy is:",accuracy_score(y_test,y_pred))
print("\n<------Classificationn Report------->\n",classification_report(y_test,y_pred))

#Predicted probabilities for the positive class
y_pred_prob = best_model.predict_proba(X_test)[:,1]
print("ROC-AUC score:", roc_auc_score(y_test,y_pred_prob))

In [ ]:
# Vizualizing ROC_AUC curve
import matplotlib.pyplot as plt 
from sklearn.metrics import roc_curve, roc_auc_score

#Computing Roc Curve
fpr, tpr, threshold = roc_curve(y_test, y_pred_prob)

# Plot ROC Curve
plt.figure(figsize=(8,8))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc_score(y_test, y_pred_prob):.2f})")
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression (Best Model)")
plt.legend()
plt.show()

```
5. Interpretation:
a. Interpret the coefficients of the logistic regression model.
b. Discuss the significance of features in predicting the target variable (survival probability in this case).
```

In [ ]:
import pandas as pd

coefficients = best_model.coef_[0] 

features = [col for col in dataset.columns.tolist() if not col == 'Outcome']

coef_df = pd.DataFrame( {
    "Feature": features,
    "Coefficient (β)": coefficients,
    "Odds Ratio (exp(β))": np.exp(coefficients)
})

print(coef_df)

## Disscussing the significance of features
```
1. Pregnancies (β=0.317, OR≈1.37): Each extra pregnancy increases the odds of diabetes by 37%, keeping other factors constant.

2. Glucose (β=0.947, OR≈2.58): The most important predictor here — each unit increase in glucose multiplies the odds of diabetes by 2.6.

3. BloodPressure, SkinThickness, Insulin (β=0.0, OR=1.0): The model found no predictive power here, maybe due to scaling, collinearity, or regularization.

4. BMI (β=0.488, OR≈1.63): Higher BMI strongly increases odds of diabetes (63% increase per unit).

5. DiabetesPedigreeFunction (β=0.143, OR≈1.15): A genetic/family risk factor — modest but positive effect.

6. Age (β=0.026, OR≈1.03): Each extra year increases odds by 3%.
```

## Deployment Locally:

```
6. Deployment with Streamlit:
In this task, you will deploy your logistic regression model using Streamlit. The deployment can be done locally. Your task includes creating a Streamlit app in Python that involves loading your trained model and setting up user inputs for predictions.
```

In [ ]:
import pickle

with open("log_reg_diabetes.pkl", "wb") as file:
    pickle.dump(best_model, file)